In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TP3-Retail-Intelligence")
    .master("local[*]")
    .getOrCreate()
)

print("Version Spark :", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/18 15:28:09 WARN Utils: Your hostname, LAPTOP-S0JER4NB, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/18 15:28:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/home/math34/projets/ipssi-archi-bigdata/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/18 15:28:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/11/18 15:28:11 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/11/18 15:28:11 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Version Spark : 4.3.0


In [2]:
# Prefixe file:// : lire sur le disque local et non dans HDFS
# (fs.defaultFS = hdfs://localhost:9000 dans la config Hadoop du J2)
# A ADAPTER : remplacer par le chemin absolu de votre copie des donnees TPCx-BB,
# en gardant le prefixe file:// (3 slashs : file:// + /home/...)
# Format attendu : file:///<dossier_donnees>/retail_project/data/store_sales/store_sales.dat
chemin_fichier = "file:///home/math34/tp_bigdata/extract/retail_project/data/store_sales/store_sales.dat"

df = (
    spark.read
    .option("header", False)
    .option("inferSchema", True)
    .option("sep", "|")
    .csv(chemin_fichier)
)

df.show(5)
df.printSchema()

+-----+-----+-----+-----+-------+----+-----+---+---+---+----+-----+------+-----+-------+-------+-------+-------+------+------+-------+-------+-------+
|  _c0|  _c1|  _c2|  _c3|    _c4| _c5|  _c6|_c7|_c8|_c9|_c10| _c11|  _c12| _c13|   _c14|   _c15|   _c16|   _c17|  _c18|  _c19|   _c20|   _c21|   _c22|
+-----+-----+-----+-----+-------+----+-----+---+---+---+----+-----+------+-----+-------+-------+-------+-------+------+------+-------+-------+-------+
|36890|71480|10001|44404| 493944|2898|44093| 10|294|  0|  31| 18.0| 30.78|23.08|  238.7| 715.48|  558.0| 954.18| 21.46|   0.0| 715.48| 736.94| 157.48|
|36890|71480|10000|44404| 493944|2898|44093| 10|279|  0|  54|99.21|148.82|83.34|3535.92|4500.36|5357.34|8036.28|225.02|   0.0|4500.36|4725.38|-856.98|
|36890|71480|15556|44404| 493944|2898|44093| 10|114|  0|  39|79.49| 94.59|83.24| 442.65|3246.36|3100.11|3689.01|184.07|616.81|2629.55|2813.62|-470.56|
|36890|45392| 4597|89832|1774847|6266|16285|  4|285|  1|  20|74.38|136.12|133.4|   54.4| 2668.

In [3]:
df_hdfs = (
    spark.read
    .option("header", False)
    .option("inferSchema", True)
    .option("sep", "|")
    .csv("hdfs:///retail-lake/raw/tpcxbb/initial/store_sales.dat")
)

df_hdfs.show(5)


+-----+-----+-----+-----+-------+----+-----+---+---+---+----+-----+------+-----+-------+-------+-------+-------+------+------+-------+-------+-------+
|  _c0|  _c1|  _c2|  _c3|    _c4| _c5|  _c6|_c7|_c8|_c9|_c10| _c11|  _c12| _c13|   _c14|   _c15|   _c16|   _c17|  _c18|  _c19|   _c20|   _c21|   _c22|
+-----+-----+-----+-----+-------+----+-----+---+---+---+----+-----+------+-----+-------+-------+-------+-------+------+------+-------+-------+-------+
|36890|71480|10001|44404| 493944|2898|44093| 10|294|  0|  31| 18.0| 30.78|23.08|  238.7| 715.48|  558.0| 954.18| 21.46|   0.0| 715.48| 736.94| 157.48|
|36890|71480|10000|44404| 493944|2898|44093| 10|279|  0|  54|99.21|148.82|83.34|3535.92|4500.36|5357.34|8036.28|225.02|   0.0|4500.36|4725.38|-856.98|
|36890|71480|15556|44404| 493944|2898|44093| 10|114|  0|  39|79.49| 94.59|83.24| 442.65|3246.36|3100.11|3689.01|184.07|616.81|2629.55|2813.62|-470.56|
|36890|45392| 4597|89832|1774847|6266|16285|  4|285|  1|  20|74.38|136.12|133.4|   54.4| 2668.

In [4]:
import os

# Test d'ecriture dans un repertoire de sortie dedie (jamais dans les sources)
# Le notebook est dans preuves/environnement/ : on remonte de 2 niveaux
# pour ecrire dans J3_Retail_Intelligence/data/test/
chemin_test = "file://" + os.path.abspath("../../data/test/ecriture_test")

df.limit(10).write.mode("overwrite").csv(chemin_test, header=True)

# Relecture pour verifier que l'ecriture a fonctionne
df_test = spark.read.option("header", True).csv(chemin_test)
print("Ecrit dans :", chemin_test)
print("Lignes relues :", df_test.count())

Ecrit dans : file:///home/math34/projets/ipssi-archi-bigdata/J3_Retail_Intelligence/data/test/ecriture_test


Lignes relues : 10
